# HridAI: train the screening classifier on CardioPHON

**Run this on Colab:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.
It should take about 20 to 30 minutes (about 500 MB of audio).

What it does:
1. Downloads CirCor and the CardioPHON encoder.
2. Turns each recording into a CardioPHON embedding, using the backend's own preprocessing.
3. Trains a small classifier. The encoder stays frozen.
4. Chooses the threshold on validation data for sensitivity >= 0.85 on MV + AV.
5. Exports `model.onnx` and `model_meta.json`, then downloads `hridai_model.zip`.

Put the zip's contents in `backend/models/`.

**Limits to state in the demo:** CirCor is pediatric and recorded with a stethoscope, so these
numbers do not describe adults or phone recordings.

Generated by `ml/build_notebook.py`. Edit that file, not this notebook.

In [ ]:
# Generated from backend/. Do not edit here.
SHARED_SOURCES = {
    'app/core/audio_config.py': 'from dataclasses import dataclass\n\n\n@dataclass(frozen=True)\nclass AudioConfig:\n    """Shared by training and inference. Values follow CardioPHON\'s config_4k.yaml."""\n\n    sample_rate: int = 4000\n    trim_seconds: float = 1.0\n    n_fft: int = 512\n    hop_length: int = 160\n    n_mels: int = 64\n    f_min: float = 60.0\n    f_max: float = 2000.0\n    log_offset: float = 1.1920929e-07\n    norm_mean: float = -4.1357055\n    norm_std: float = 2.863227\n\n\nAUDIO = AudioConfig()\n',
    'app/services/audio.py': 'from math import gcd\n\nimport numpy as np\nfrom scipy.signal import resample_poly\n\nfrom app.core.audio_config import AUDIO\n\n\ndef preprocess(audio: np.ndarray, sample_rate: int) -> np.ndarray:\n    """Resamples, trims handling noise and peak-normalizes mono audio."""\n    audio = np.asarray(audio, dtype=np.float32)\n    if sample_rate != AUDIO.sample_rate:\n        divisor = gcd(sample_rate, AUDIO.sample_rate)\n        audio = resample_poly(audio, AUDIO.sample_rate // divisor, sample_rate // divisor)\n\n    trim = int(AUDIO.trim_seconds * AUDIO.sample_rate)\n    if audio.size <= 2 * trim:\n        raise ValueError("Recording is too short to trim")\n    audio = audio[trim:-trim]\n\n    peak = np.abs(audio).max()\n    return (audio / peak if peak > 0 else audio).astype(np.float32)\n',
    'app/services/features.py': 'from functools import lru_cache\n\nimport numpy as np\nfrom numpy.lib.stride_tricks import sliding_window_view\nfrom scipy.signal import get_window\n\nfrom app.core.audio_config import AUDIO\n\n_MEL_BREAK_HZ = 1000.0\n_MEL_LINEAR_SLOPE = 200.0 / 3\n_MEL_BREAK = _MEL_BREAK_HZ / _MEL_LINEAR_SLOPE\n_MEL_LOG_STEP = np.log(6.4) / 27\n\n\ndef _hz_to_mel(hz: float) -> float:\n    if hz < _MEL_BREAK_HZ:\n        return hz / _MEL_LINEAR_SLOPE\n    return _MEL_BREAK + np.log(hz / _MEL_BREAK_HZ) / _MEL_LOG_STEP\n\n\ndef _mel_to_hz(mels: np.ndarray) -> np.ndarray:\n    return np.where(\n        mels < _MEL_BREAK,\n        mels * _MEL_LINEAR_SLOPE,\n        _MEL_BREAK_HZ * np.exp((mels - _MEL_BREAK) * _MEL_LOG_STEP),\n    )\n\n\n@lru_cache(maxsize=1)\ndef _mel_filterbank() -> np.ndarray:\n    """Slaney-style filters, matching librosa and nnAudio defaults."""\n    fft_hz = np.fft.rfftfreq(AUDIO.n_fft, 1 / AUDIO.sample_rate)\n    mel_points = np.linspace(_hz_to_mel(AUDIO.f_min), _hz_to_mel(AUDIO.f_max), AUDIO.n_mels + 2)\n    edges_hz = _mel_to_hz(mel_points)\n\n    lower, center, upper = edges_hz[:-2, None], edges_hz[1:-1, None], edges_hz[2:, None]\n    rising = (fft_hz - lower) / (center - lower)\n    falling = (upper - fft_hz) / (upper - center)\n    filters = np.maximum(0, np.minimum(rising, falling))\n    return (2 / (upper - lower)) * filters\n\n\ndef log_mel(audio: np.ndarray) -> np.ndarray:\n    """Returns normalized log-mel features shaped [n_mels, frames]."""\n    audio = np.asarray(audio, dtype=np.float32)\n    if audio.ndim != 1 or audio.size < AUDIO.n_fft:\n        raise ValueError("Expected mono audio of at least one FFT window")\n\n    half_window = AUDIO.n_fft // 2\n    padded = np.pad(audio, half_window, mode="reflect")\n    frames = sliding_window_view(padded, AUDIO.n_fft)[:: AUDIO.hop_length]\n    window = get_window("hann", AUDIO.n_fft, fftbins=True)\n    power = np.abs(np.fft.rfft(frames * window, axis=1)) ** 2\n\n    mel = _mel_filterbank() @ power.T\n    log_values = np.log(mel + AUDIO.log_offset)\n    return ((log_values - AUDIO.norm_mean) / AUDIO.norm_std).astype(np.float32)\n\n\ndef model_input(audio: np.ndarray) -> np.ndarray:\n    """Returns the model\'s float32 input shaped [1, 1, n_mels, frames]."""\n    return log_mel(audio)[np.newaxis, np.newaxis]\n',
}

## 1. Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

WORK = Path(os.environ.get("HRIDAI_WORK", "/content/hridai"))
LIMIT_PATIENTS = int(os.environ.get("HRIDAI_LIMIT_PATIENTS", "0")) or None  # 0 = all patients
CHECKPOINT = os.environ.get("HRIDAI_CHECKPOINT", "")
MODEL_VERSION = "cardiophon-ft-0.1"
SEED = 42
TARGET_SENSITIVITY = 0.85
DEMO_SITES = ("MV", "AV")

WORK.mkdir(parents=True, exist_ok=True)
if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "soundfile"],
        check=True,
    )

for relative_path, source in SHARED_SOURCES.items():
    target = WORK / "src" / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
for package in ("app", "app/core", "app/services"):
    (WORK / "src" / package / "__init__.py").touch()
sys.path.insert(0, str(WORK / "src"))

import numpy as np
import onnxruntime as ort
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn

from app.core.audio_config import AUDIO
from app.services.audio import preprocess
from app.services.features import model_input

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 2. Download CirCor

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen, urlretrieve

CIRCOR = "https://physionet-open.s3.amazonaws.com/circor-heart-sound/1.0.3"
DATA = WORK / "circor"
(DATA / "training_data").mkdir(parents=True, exist_ok=True)

urlretrieve(f"{CIRCOR}/training_data.csv", DATA / "training_data.csv")
with urlopen(f"{CIRCOR}/RECORDS") as response:
    all_records = response.read().decode().split()

patients = pd.read_csv(DATA / "training_data.csv")
patients = patients[patients["Murmur"] != "Unknown"]
if LIMIT_PATIENTS:
    patients = patients.groupby("Murmur", group_keys=False).sample(
        n=LIMIT_PATIENTS // 2, random_state=SEED
    )

wanted_ids = {str(patient_id) for patient_id in patients["Patient ID"]}
records = [r for r in all_records if Path(r).name.split("_")[0] in wanted_ids]


def fetch(record):
    target = DATA / f"{record}.wav"
    if target.exists() and target.stat().st_size > 0:
        return
    for attempt in range(3):
        try:
            urlretrieve(f"{CIRCOR}/{record}.wav", target)
            return
        except OSError:
            if attempt == 2:
                raise


with ThreadPoolExecutor(16) as pool:
    list(pool.map(fetch, records))
print(f"{len(records)} recordings from {len(patients)} patients")

## 3. Labels and patient-level split

In [ ]:
murmur = patients.set_index("Patient ID")
rows = []
for record in records:
    patient_id, site = Path(record).name.split("_")[:2]
    patient_id = int(patient_id)
    if site not in ("AV", "MV", "PV", "TV"):
        continue
    # The Murmur column is per patient, so a site is positive only if the murmur was heard there.
    sites_with_murmur = str(murmur.loc[patient_id, "Murmur locations"]).split("+")
    label = int(murmur.loc[patient_id, "Murmur"] == "Present" and site in sites_with_murmur)
    rows.append({"record": record, "patient": patient_id, "site": site, "label": label})
recordings = pd.DataFrame(rows)

patient_ids = murmur.index.to_numpy()
patient_labels = (murmur["Murmur"] == "Present").astype(int).to_numpy()
train_ids, rest_ids, _, rest_labels = train_test_split(
    patient_ids, patient_labels, test_size=0.30, stratify=patient_labels, random_state=SEED
)
val_ids, test_ids = train_test_split(
    rest_ids, test_size=0.50, stratify=rest_labels, random_state=SEED
)
split_of = {i: "train" for i in train_ids} | {i: "val" for i in val_ids}
split_of |= {i: "test" for i in test_ids}
recordings["split"] = recordings["patient"].map(split_of)
recordings.to_csv(WORK / "splits.csv", index=False)
print(recordings.groupby("split")["label"].agg(["size", "sum"]))

## 4. Load the CardioPHON encoder

In [ ]:
def checkpoint_path():
    if CHECKPOINT:
        return CHECKPOINT
    repo = WORK / "cardiophon"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://git.lih.lu/vdespotovic/cardiophon.git",
             str(repo)],
            check=True,
        )
    return repo / "heart_sound_classification/checkpoints/pretrained_physionet_2022_4kHz.pth"


class Encoder(nn.Module):
    """CardioPHON's AudioNTT2022 encoder, with mean + max pooling over time."""

    def __init__(self, embedding_size=3072, channels=64, hidden=2048):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, channels, 3, stride=1, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(channels, channels, 3, stride=1, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
        )
        conv_size = channels * (AUDIO.n_mels // 4)
        self.fc = nn.Sequential(
            nn.Linear(conv_size, hidden),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(hidden, embedding_size - conv_size),
            nn.ReLU(),
        )

    def forward(self, log_mel):
        frames = self.features(log_mel).permute(0, 3, 2, 1).flatten(2)
        frames = torch.cat([frames, self.fc(frames)], dim=2)
        return torch.max(frames, dim=1)[0] + torch.mean(frames, dim=1)


encoder = Encoder()
encoder.load_state_dict(torch.load(checkpoint_path(), map_location="cpu", weights_only=True))
encoder = encoder.to(DEVICE).eval()

## 5. Compute embeddings

In [ ]:
PHONE_HIGH_PASS = butter(4, 120, btype="highpass", fs=AUDIO.sample_rate, output="sos")


def embed(audio, sample_rate):
    features = torch.from_numpy(model_input(preprocess(audio, sample_rate))).to(DEVICE)
    with torch.no_grad():
        return encoder(features)[0].cpu().numpy()


def load(record):
    audio, sample_rate = sf.read(DATA / f"{record}.wav", dtype="float32")
    return audio, sample_rate


embeddings, phone_embeddings = [], []
for number, record in enumerate(recordings["record"], start=1):
    audio, sample_rate = load(record)
    embeddings.append(embed(audio, sample_rate))
    # Phone mics lose low frequencies, so training also sees a rough simulation of that.
    phone_audio = sosfiltfilt(PHONE_HIGH_PASS, audio).astype(np.float32)
    phone_embeddings.append(embed(phone_audio, sample_rate))
    if number % 500 == 0:
        print(f"{number}/{len(recordings)}")
embeddings = np.stack(embeddings)
phone_embeddings = np.stack(phone_embeddings)
print("Embeddings:", embeddings.shape)

## 6. Train the classifier

In [ ]:
is_train = (recordings["split"] == "train").to_numpy()
is_val = (recordings["split"] == "val").to_numpy()
labels = recordings["label"].to_numpy(dtype=np.float32)

train_x = np.concatenate([embeddings[is_train], phone_embeddings[is_train]])
train_y = np.concatenate([labels[is_train], labels[is_train]])
feature_mean = train_x.mean(axis=0, keepdims=True)
feature_std = train_x.std(axis=0, keepdims=True) + 1e-6


def scaled(values):
    return torch.from_numpy((values - feature_mean) / feature_std).float().to(DEVICE)


head = nn.Sequential(
    nn.Linear(embeddings.shape[1], 256),
    nn.Dropout(0.5),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.Dropout(0.5),
    nn.ReLU(),
    nn.Linear(256, 1),
).to(DEVICE)

positive_weight = torch.tensor((train_y == 0).sum() / max((train_y == 1).sum(), 1)).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(head.parameters(), lr=3e-4, weight_decay=1e-4)
inputs, targets = scaled(train_x), torch.from_numpy(train_y).to(DEVICE)


def probabilities(values):
    head.eval()
    with torch.no_grad():
        return torch.sigmoid(head(scaled(values)))[:, 0].cpu().numpy()


best_auc, best_state, stale_epochs = -1.0, None, 0
for epoch in range(1, 61):
    head.train()
    for batch in torch.randperm(len(inputs), device=DEVICE).split(32):
        optimizer.zero_grad()
        loss = loss_fn(head(inputs[batch])[:, 0], targets[batch])
        loss.backward()
        optimizer.step()

    val_auc = roc_auc_score(labels[is_val], probabilities(embeddings[is_val]))
    if val_auc > best_auc:
        best_auc, stale_epochs = val_auc, 0
        best_state = {name: value.clone() for name, value in head.state_dict().items()}
    else:
        stale_epochs += 1
    if epoch % 5 == 0:
        print(f"epoch {epoch}: val recording AUC {val_auc:.3f}")
    if stale_epochs >= 10:
        break
head.load_state_dict(best_state)
print(f"Best val recording AUC: {best_auc:.3f}")

## 7. Choose the threshold and test

In [ ]:
def patient_scores(split, sites):
    """Matches the backend: mean per site, then max over sites."""
    chosen = recordings[(recordings["split"] == split) & recordings["site"].isin(sites)].copy()
    chosen["probability"] = probabilities(embeddings[chosen.index])
    per_site = chosen.groupby(["patient", "site"]).agg(
        probability=("probability", "mean"), label=("label", "max")
    )
    per_patient = per_site.groupby("patient").agg(
        probability=("probability", "max"), label=("label", "max")
    )
    return per_patient["probability"].to_numpy(), per_patient["label"].to_numpy()


def choose_threshold(scores, truth):
    """Highest threshold that still reaches the target sensitivity."""
    for threshold in sorted(set(scores.round(4)), reverse=True):
        if (scores[truth == 1] >= threshold).mean() >= TARGET_SENSITIVITY:
            return float(threshold)
    return float(scores.min())


def report(split, sites, threshold):
    scores, truth = patient_scores(split, sites)
    tn, fp, fn, tp = confusion_matrix(truth, scores >= threshold, labels=[0, 1]).ravel()
    return {
        "patients": len(truth),
        "sensitivity": round(tp / max(tp + fn, 1), 3),
        "specificity": round(tn / max(tn + fp, 1), 3),
        "auc": round(float(roc_auc_score(truth, scores)), 3),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }


threshold = choose_threshold(*patient_scores("val", DEMO_SITES))
results = {
    "threshold": threshold,
    "val_mv_av": report("val", DEMO_SITES, threshold),
    "test_mv_av": report("test", DEMO_SITES, threshold),
    "test_mv_only": report("test", ("MV",), threshold),
    "test_av_only": report("test", ("AV",), threshold),
}
print(json.dumps(results, indent=2))

## 8. Export

In [ ]:
import shutil


class ScreeningModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder
        self.head = head
        self.register_buffer("feature_mean", torch.from_numpy(feature_mean).float())
        self.register_buffer("feature_std", torch.from_numpy(feature_std).float())

    def forward(self, log_mel):
        scaled_embedding = (self.encoder(log_mel) - self.feature_mean) / self.feature_std
        return self.head(scaled_embedding)[:, 0]


OUTPUT = WORK / "hridai_model"
OUTPUT.mkdir(exist_ok=True)
model = ScreeningModel().to("cpu").eval()
torch.onnx.export(
    model,
    (torch.zeros(1, 1, AUDIO.n_mels, 326),),
    OUTPUT / "model.onnx",
    input_names=["log_mel"],
    output_names=["logit"],
    dynamic_axes={"log_mel": {0: "batch", 3: "frames"}, "logit": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

session = ort.InferenceSession(OUTPUT / "model.onnx", providers=["CPUExecutionProvider"])
golden_audio, golden_rate = load(recordings[recordings["split"] == "test"]["record"].iloc[0])
golden_clean = preprocess(golden_audio, golden_rate)
golden_input = model_input(golden_clean)
onnx_logit = float(session.run(None, {"log_mel": golden_input})[0][0])
with torch.no_grad():
    torch_logit = float(model(torch.from_numpy(golden_input))[0])
assert abs(onnx_logit - torch_logit) < 1e-3, "ONNX output differs from PyTorch"

np.savez_compressed(OUTPUT / "golden.npz", audio=golden_clean, logit=onnx_logit)
(OUTPUT / "model_meta.json").write_text(
    json.dumps(
        {
            "model_version": MODEL_VERSION,
            "output": "logit",
            "threshold": threshold,
            "sample_rate": AUDIO.sample_rate,
            "n_mels": AUDIO.n_mels,
            "n_fft": AUDIO.n_fft,
            "hop_length": AUDIO.hop_length,
            "f_min": AUDIO.f_min,
            "f_max": AUDIO.f_max,
            "results": results,
        },
        indent=2,
    )
)
recordings.to_csv(OUTPUT / "splits.csv", index=False)
archive = shutil.make_archive(str(WORK / "hridai_model"), "zip", OUTPUT)
print("Saved:", archive)

if "google.colab" in sys.modules:
    from google.colab import files

    files.download(archive)